<a href="https://colab.research.google.com/github/rakib181/MachineLearningNLP/blob/main/Assessment_01(Part_B).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import sklearn
import nltk as nl
import operator
import matplotlib.pyplot as plt

In [2]:
nl.download('stopwords')
nl.download('punkt')
nl.download('wordnet')
nl.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
url = "https://raw.githubusercontent.com/rakib181/MachineLearningNLP/refs/heads/main/brighter_intensity_q1.csv"
df = pd.read_csv(url)
df.tail()

,id,text,Emotion_Intensity
2041,2042,"And they asked me when I am getting married, a...",2
2042,2043,Better late then never!,2
2043,2044,"In the last three weeks, I have started lookin...",2
2044,2045,""" So I will remain positive for as long as I l...",2
2045,2046,`` Bella my head is fine.,2


In [4]:
lemmatizer = nl.stem.WordNetLemmatizer()

def get_list_tokens(string):
  sentence_split=nl.tokenize.sent_tokenize(string)
  list_tokens = []
  for sentence in sentence_split:
    list_tokens_sentence = nl.tokenize.word_tokenize(sentence)
    for token in list_tokens_sentence:
      list_tokens.append(lemmatizer.lemmatize(token).lower())
  return list_tokens



In [5]:
stopwords = set(nl.corpus.stopwords.words('english'))
stopwords.add(".")
stopwords.add(",")
stopwords.add("--")
stopwords.add("``")


dict_word_frequency = {}
for sentence in df['text']:
  sentence_tokens = get_list_tokens(sentence)
  for token in sentence_tokens:
    if token not in stopwords:
      if token in dict_word_frequency:
        dict_word_frequency[token] += 1
      else:
        dict_word_frequency[token] = 1



vocabulary = []
for word,frequency in dict_word_frequency.items():
  vocabulary.append(word)

In [6]:
def get_vector_text(list_vocab,string):
  vector_text = np.zeros(len(list_vocab))
  list_tokens_string=get_list_tokens(string)
  for i, word in enumerate(list_vocab):
    if word in list_tokens_string:
      vector_text[i] = list_tokens_string.count(word)
  return vector_text

In [7]:
X_train = []
Y_train = []
for sentence, intensity in zip(df["text"], df["Emotion_Intensity"]):
  vector_text = get_vector_text(vocabulary, sentence)
  X_train.append(vector_text)
  Y_train.append(intensity)

In [8]:
X = np.array(X_train)
y = np.array(Y_train)

In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42
)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

Training features shape: (1636, 4573)
Testing features shape: (410, 4573)


In [10]:
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC

svm_model = SVC(kernel = "linear", gamma = "auto")
svm_model.fit(X_train, y_train)
y_pred = svm_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {accuracy * 100:.2f}%")

Overall Accuracy: 61.71%


In [31]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

dt_model = DecisionTreeRegressor(random_state = 42)
dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)
print("RMSE : ", mean_squared_error(y_test, y_pred_dt) ** 0.5)


RMSE :  0.8568234015614972


In [32]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
y_pred_rf = model.predict(X_test)
print("RMSE : ", mean_squared_error(y_test, y_pred_rf) ** 0.5)

RMSE :  0.7213926233842225


In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)
y_pred_lr = model.predict(X_test)
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred_lr) * 100:.2f}%")



Overall Accuracy: 58.29%
